# CFTC Disaggregated Report — Petroleum Complex Overview

Basic interrogation of the disaggregated combined report
to understand what's inside for petroleum products.

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = '/Users/oualid/Documents/Projects/omroot_repos/cftc_data/cftc_downloads/disaggregated_combined.csv'
raw = pd.read_csv(RAW_PATH, low_memory=False)

print(f'Shape: {raw.shape}')
print(f'Columns: {raw.shape[1]}')
dates = pd.to_datetime(raw['report_date_as_yyyy_mm_dd'])
print(f'Date range: {dates.min().date()} -> {dates.max().date()}')
print(f'Unique markets: {raw["contract_market_name"].nunique()}')
print(f'Unique commodities: {raw["commodity_name"].nunique()}')

Shape: (184173, 194)
Columns: 194
Date range: 2006-06-13 -> 2026-03-24
Unique markets: 665
Unique commodities: 58


## 1. Commodity groups in the report

In [2]:
# All commodity groups
groups = raw.groupby('commodity_group_name')['contract_market_name'].nunique().sort_values(ascending=False)
print('Commodity groups (by number of contracts):')
print(groups.to_string())

Commodity groups (by number of contracts):
commodity_group_name
NATURAL RESOURCES    624
AGRICULTURE           41


## 2. Petroleum complex: all contracts

In [3]:
PETRO_COMMODITIES = ['CRUDE OIL', 'HEATING OIL-DIESEL-GASOIL', 'GASOLINE']

petro = raw[raw['commodity_name'].isin(PETRO_COMMODITIES)].copy()
petro['date'] = pd.to_datetime(petro['report_date_as_yyyy_mm_dd'])

print(f'Petroleum rows: {len(petro)} ({len(petro)/len(raw)*100:.1f}% of total)')
print(f'\nContracts by commodity:')

for comm in PETRO_COMMODITIES:
    sub = petro[petro['commodity_name'] == comm]
    contracts = sub.groupby('contract_market_name').agg(
        rows=('date', 'count'),
        start=('date', 'min'),
        end=('date', 'max'),
        code=('cftc_contract_market_code', 'first'),
    ).sort_values('rows', ascending=False)
    print(f'\n=== {comm} ({len(contracts)} contracts) ===')
    for name, r in contracts.iterrows():
        print(f'  {name:<45s}  code={r["code"]}  rows={r["rows"]:>5d}  '
              f'{r["start"].date()} -> {r["end"].date()}')

Petroleum rows: 16529 (9.0% of total)

Contracts by commodity:

=== CRUDE OIL (36 contracts) ===
  WTI-PHYSICAL                                   code=067651  rows= 1033  2006-06-13 -> 2026-03-24
  WTI FINANCIAL CRUDE OIL                        code=06765A  rows=  972  2007-02-20 -> 2026-03-24
  CRUDE OIL AVG PRICE OPTIONS                    code=06765C  rows=  929  2008-04-29 -> 2026-03-24
  CRUDE OIL, LIGHT SWEET-WTI                     code=067411  rows=  870  2009-07-28 -> 2026-03-24
  BRENT LAST DAY                                 code=06765T  rows=  780  2011-03-08 -> 2026-03-24
  CRUDE OIL CAL SPREAD OPTIONS                   code=067657  rows=  741  2008-04-29 -> 2026-02-17
  EUR STYLE CRUDE OIL OPTIONS                    code=06765B  rows=  520  2008-04-29 -> 2022-08-16
  CRUDE DIFF-TMX WCS 1A INDEX                    code=06742G  rows=  423  2018-02-20 -> 2026-03-24
  WTI-BRENT SPREAD OPTION                        code=06765Z  rows=  406  2013-02-26 -> 2025-04-22
  WTI  HOUST

## 3. Futures-only vs Combined

In [4]:
print('futonly_or_combined values:')
print(raw['futonly_or_combined'].value_counts().to_string())

# For petroleum, show the split
print(f'\nPetroleum:')
print(petro['futonly_or_combined'].value_counts().to_string())

# Example: WTI-PHYSICAL
wti = petro[petro['contract_market_name'] == 'WTI-PHYSICAL']
print(f'\nWTI-PHYSICAL:')
print(wti['futonly_or_combined'].value_counts().to_string())

futonly_or_combined values:
futonly_or_combined
Combined    184173

Petroleum:
futonly_or_combined
Combined    16529

WTI-PHYSICAL:
futonly_or_combined
Combined    1033


## 4. Column inventory

Group the 194 columns by purpose.

In [5]:
cols = raw.columns.tolist()

groups = {
    'Identification': [c for c in cols if any(k in c.lower() for k in
        ['market_name', 'report_date', 'contract_market', 'cftc_', 'commodity',
         'contract_units', 'subgroup', 'futonly', 'yyyy_report'])],
    'Open Interest': [c for c in cols if 'open_interest' in c.lower() and 'pct' not in c.lower()],
    'Positions (all)': [c for c in cols if 'positions' in c.lower() and '_all' in c
                        and 'pct' not in c.lower() and 'old' not in c and 'other' not in c.split('_')[-1]],
    'Positions (old/other)': [c for c in cols if 'positions' in c.lower()
                              and ('old' in c or c.endswith('_1') or c.endswith('_2'))
                              and 'pct' not in c.lower()],
    'Pct of OI': [c for c in cols if 'pct_of_oi' in c.lower() or 'pct_of_open' in c.lower()],
    'Traders count': [c for c in cols if 'traders_' in c.lower()],
    'Concentration': [c for c in cols if 'conc_' in c.lower()],
    'Changes': [c for c in cols if 'change_in' in c.lower()],
}

classified = set()
for g, cs in groups.items():
    classified.update(cs)
groups['Other'] = [c for c in cols if c not in classified]

for g, cs in groups.items():
    print(f'\n{g} ({len(cs)} columns):')
    for c in cs[:8]:
        print(f'  {c}')
    if len(cs) > 8:
        print(f'  ... and {len(cs)-8} more')


Identification (14 columns):
  report_date_as_yyyy_mm_dd
  yyyy_report_week_ww
  contract_market_name
  cftc_contract_market_code
  cftc_market_code
  cftc_region_code
  cftc_commodity_code
  commodity_name
  ... and 6 more

Open Interest (4 columns):
  open_interest_all
  open_interest_old
  open_interest_other
  change_in_open_interest_all

Positions (all) (8 columns):
  swap_positions_long_all
  swap__positions_short_all
  swap__positions_spread_all
  m_money_positions_long_all
  m_money_positions_short_all
  tot_rept_positions_long_all
  nonrept_positions_long_all
  nonrept_positions_short_all

Positions (old/other) (22 columns):
  prod_merc_positions_long_1
  prod_merc_positions_short_1
  swap_positions_long_old
  swap__positions_short_old
  swap__positions_spread_old
  m_money_positions_long_old
  m_money_positions_short_old
  m_money_positions_spread_1
  ... and 14 more

Pct of OI (48 columns):
  pct_of_open_interest_all
  pct_of_oi_prod_merc_long
  pct_of_oi_prod_merc_short
  

## 5. The 5 trader categories — column mapping

In [6]:
CATEGORIES = {
    'Producer/Merchant': {
        'long': 'prod_merc_positions_long',
        'short': 'prod_merc_positions_short',
    },
    'Swap Dealers': {
        'long': 'swap_positions_long_all',
        'short': 'swap__positions_short_all',  # note double underscore
        'spread': 'swap__positions_spread_all',
    },
    'Managed Money': {
        'long': 'm_money_positions_long_all',
        'short': 'm_money_positions_short_all',
        'spread': 'm_money_positions_spread',
    },
    'Other Reportables': {
        'long': 'other_rept_positions_long',
        'short': 'other_rept_positions_short',
        'spread': 'other_rept_positions_spread',
    },
    'Non-Reportables': {
        'long': 'nonrept_positions_long_all',
        'short': 'nonrept_positions_short_all',
    },
}

print('Trader category column mapping:')
print()
for cat, cols_map in CATEGORIES.items():
    print(f'{cat}:')
    for side, col in cols_map.items():
        exists = col in raw.columns
        print(f'  {side:8s} -> {col}  {"OK" if exists else "MISSING!"}')
    print()

Trader category column mapping:

Producer/Merchant:
  long     -> prod_merc_positions_long  OK
  short    -> prod_merc_positions_short  OK

Swap Dealers:
  long     -> swap_positions_long_all  OK
  short    -> swap__positions_short_all  OK
  spread   -> swap__positions_spread_all  OK

Managed Money:
  long     -> m_money_positions_long_all  OK
  short    -> m_money_positions_short_all  OK
  spread   -> m_money_positions_spread  OK

Other Reportables:
  long     -> other_rept_positions_long  OK
  short    -> other_rept_positions_short  OK
  spread   -> other_rept_positions_spread  OK

Non-Reportables:
  long     -> nonrept_positions_long_all  OK
  short    -> nonrept_positions_short_all  OK



## 6. Primary petroleum contracts

In [7]:
PRIMARY = {
    'WTI':  'WTI-PHYSICAL',
    'HO':   'NY HARBOR ULSD',
    'RBOB': 'GASOLINE RBOB',
}

print(f'{"Ticker":<6s}  {"Contract":<25s}  {"Code":<10s}  {"Rows":>5s}  {"Type":<10s}  Date range')
print('-' * 90)

for ticker, name in PRIMARY.items():
    sub = petro[petro['contract_market_name'] == name]
    code = sub['cftc_contract_market_code'].iloc[0] if len(sub) > 0 else '?'
    ftype = sub['futonly_or_combined'].iloc[0] if len(sub) > 0 else '?'
    d_min = sub['date'].min().date() if len(sub) > 0 else '?'
    d_max = sub['date'].max().date() if len(sub) > 0 else '?'
    print(f'{ticker:<6s}  {name:<25s}  {code:<10s}  {len(sub):>5d}  {ftype:<10s}  {d_min} -> {d_max}')

print()
print('Note: Brent and Gasoil are ICE Futures Europe products,')
print('not in the CFTC disaggregated report. Use ice_cot.csv instead.')

Ticker  Contract                   Code         Rows  Type        Date range


------------------------------------------------------------------------------------------
WTI     WTI-PHYSICAL               067651       1033  Combined    2006-06-13 -> 2026-03-24
HO      NY HARBOR ULSD             022651       1033  Combined    2006-06-13 -> 2026-03-24
RBOB    GASOLINE RBOB              111659       1033  Combined    2006-06-13 -> 2026-03-24

Note: Brent and Gasoil are ICE Futures Europe products,
not in the CFTC disaggregated report. Use ice_cot.csv instead.


## 7. Data quality checks

In [8]:
for ticker, name in PRIMARY.items():
    sub = petro[petro['contract_market_name'] == name].copy()
    print(f'=== {ticker} ({name}) ===')
    
    # Missing values in key columns
    key_cols = ['open_interest_all', 'prod_merc_positions_long',
                'm_money_positions_long_all', 'report_date_as_yyyy_mm_dd']
    for c in key_cols:
        n_miss = sub[c].isna().sum()
        if n_miss > 0:
            print(f'  Missing {c}: {n_miss}')
    
    # Duplicate dates
    n_dup = sub['date'].duplicated().sum()
    print(f'  Duplicate dates: {n_dup}')
    
    # Zero OI
    oi = pd.to_numeric(sub['open_interest_all'], errors='coerce')
    n_zero = (oi == 0).sum()
    print(f'  Zero OI weeks: {n_zero}')
    
    # Date gaps (weeks with no report)
    dates_sorted = sub['date'].sort_values()
    diffs = dates_sorted.diff().dt.days
    big_gaps = diffs[diffs > 10]
    print(f'  Date gaps > 10 days: {len(big_gaps)}')
    if len(big_gaps) > 0:
        for idx in big_gaps.index[:3]:
            print(f'    {dates_sorted.iloc[dates_sorted.index.get_loc(idx)-1].date()} -> '
                  f'{dates_sorted.loc[idx].date()} ({int(diffs.loc[idx])} days)')
    
    # Day-of-week distribution
    dow = sub['date'].dt.day_name().value_counts()
    print(f'  Day-of-week: {dow.to_dict()}')
    print()

=== WTI (WTI-PHYSICAL) ===
  Duplicate dates: 0
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 1019, 'Monday': 13, 'Wednesday': 1}

=== HO (NY HARBOR ULSD) ===
  Duplicate dates: 0
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 1019, 'Monday': 13, 'Wednesday': 1}

=== RBOB (GASOLINE RBOB) ===
  Duplicate dates: 0
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 1019, 'Monday': 13, 'Wednesday': 1}



## 8. Position identity check

Verify: long positions across all categories should sum to total.
Same for short.

In [9]:
wti = petro[petro['contract_market_name'] == 'WTI-PHYSICAL'].copy()

for c in ['open_interest_all', 'prod_merc_positions_long', 'prod_merc_positions_short',
          'swap_positions_long_all', 'swap__positions_short_all',
          'm_money_positions_long_all', 'm_money_positions_short_all',
          'other_rept_positions_long', 'other_rept_positions_short',
          'nonrept_positions_long_all', 'nonrept_positions_short_all',
          'tot_rept_positions_long_all', 'tot_rept_positions_short',
          'swap__positions_spread_all', 'm_money_positions_spread',
          'other_rept_positions_spread']:
    wti[c] = pd.to_numeric(wti[c], errors='coerce')

# Total long = sum of all category longs + spreads (spread counted in both long and short)
# Actually: OI = reportable long + non-reportable long = reportable short + non-reportable short
# And: total_reportable_long = PM_long + SD_long + SD_spread + MM_long + MM_spread + OR_long + OR_spread

wti['sum_rept_long'] = (wti['prod_merc_positions_long']
                        + wti['swap_positions_long_all'] + wti['swap__positions_spread_all']
                        + wti['m_money_positions_long_all'] + wti['m_money_positions_spread']
                        + wti['other_rept_positions_long'] + wti['other_rept_positions_spread'])

wti['sum_rept_short'] = (wti['prod_merc_positions_short']
                         + wti['swap__positions_short_all'] + wti['swap__positions_spread_all']
                         + wti['m_money_positions_short_all'] + wti['m_money_positions_spread']
                         + wti['other_rept_positions_short'] + wti['other_rept_positions_spread'])

# Check: sum_rept_long == tot_rept_positions_long_all?
diff_long = (wti['sum_rept_long'] - wti['tot_rept_positions_long_all']).abs()
diff_short = (wti['sum_rept_short'] - wti['tot_rept_positions_short']).abs()

print('WTI-PHYSICAL position identity check:')
print(f'  Reportable long  — max diff: {diff_long.max():.0f}, mean diff: {diff_long.mean():.1f}')
print(f'  Reportable short — max diff: {diff_short.max():.0f}, mean diff: {diff_short.mean():.1f}')

# OI = reportable + non-reportable (long side)
wti['sum_all_long'] = wti['tot_rept_positions_long_all'] + wti['nonrept_positions_long_all']
diff_oi_long = (wti['sum_all_long'] - wti['open_interest_all']).abs()
print(f'  OI = rept_long + nonrept_long — max diff: {diff_oi_long.max():.0f}')

wti['sum_all_short'] = wti['tot_rept_positions_short'] + wti['nonrept_positions_short_all']
diff_oi_short = (wti['sum_all_short'] - wti['open_interest_all']).abs()
print(f'  OI = rept_short + nonrept_short — max diff: {diff_oi_short.max():.0f}')

WTI-PHYSICAL position identity check:
  Reportable long  — max diff: 2, mean diff: 0.6
  Reportable short — max diff: 3, mean diff: 0.5
  OI = rept_long + nonrept_long — max diff: 1
  OI = rept_short + nonrept_short — max diff: 1


## 9. Sample data: WTI-PHYSICAL

In [10]:
sample_cols = [
    'report_date_as_yyyy_mm_dd',
    'open_interest_all',
    'prod_merc_positions_long', 'prod_merc_positions_short',
    'swap_positions_long_all', 'swap__positions_short_all',
    'm_money_positions_long_all', 'm_money_positions_short_all',
    'other_rept_positions_long', 'other_rept_positions_short',
    'nonrept_positions_long_all', 'nonrept_positions_short_all',
]

sample = wti[sample_cols].tail(5)
print('Last 5 weeks of WTI-PHYSICAL:')
print(sample.to_string(index=False))

Last 5 weeks of WTI-PHYSICAL:
report_date_as_yyyy_mm_dd  open_interest_all  prod_merc_positions_long  prod_merc_positions_short  swap_positions_long_all  swap__positions_short_all  m_money_positions_long_all  m_money_positions_short_all  other_rept_positions_long  other_rept_positions_short  nonrept_positions_long_all  nonrept_positions_short_all
  2026-02-24T00:00:00.000            2724561                    611101                     468313                   137950                     570193                      191809                        93622                     166794                       24016                      116058                        67567
  2026-03-03T00:00:00.000            2889103                    635467                     443074                   146457                     640705                      179794                        71373                     182264                       42716                      108237                        54350
  2026-03-10T